# Steel Defect Segmentation — YOLO26s-seg training (Colab)

Trains `yolo26s-seg` (imgsz=1024) on the Severstal Steel Defect Detection dataset converted to YOLO-seg format.

**Before Runtime → Run all:**
1. Runtime → Change runtime type → **GPU** (T4 works, A100 is faster).
2. Set `REPO_URL` in the config cell below to this project's GitHub URL.
3. Kaggle credentials — get one from kaggle.com → profile → Settings → API. Any ONE of these works:
   - **Colab Secret `KAGGLE_API_TOKEN`** (current Kaggle API token — recommended, key icon in left sidebar, enable notebook access); or
   - **Colab Secrets `KAGGLE_USERNAME` + `KAGGLE_KEY`** (legacy key pair); or
   - upload the token as `MyDrive/kaggle_access_token.txt` (just the raw token string); or
   - upload a legacy `kaggle.json` to Google Drive root (`MyDrive/kaggle.json`).
4. You must have joined the [Severstal competition](https://www.kaggle.com/competitions/severstal-steel-defect-detection) and accepted its rules.

Checkpoints are written to Drive every epoch, so if the session disconnects just **Run all again — training auto-resumes from `last.pt`**.

When training finishes, `best.pt` and `results.csv` are copied to `MyDrive/steel-defect-segmentation/final/`. Download `best.pt` into the repo's `weights/` folder for Phase 2 (evaluation / export / demo).

In [ ]:
# ---- config ----
REPO_URL = "https://github.com/tun0000/steel-defect-segmentation.git"
RUN_NAME = "yolo26s-seg-1024"
DRIVE_ROOT = "/content/drive/MyDrive/steel-defect-segmentation"
IMGSZ = 1024
EPOCHS = 100
SEED = 42

assert REPO_URL != "TODO_SET_ME", "Set REPO_URL to the project's GitHub URL first"

In [ ]:
# ---- GPU check ----
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU — set Runtime -> Change runtime type -> GPU"

In [ ]:
# ---- mount Drive (checkpoints live here so disconnects are safe) ----
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ---- Kaggle credentials: current API token first, legacy formats as fallback ----
import os, shutil
from pathlib import Path

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)


def _secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None


api_token = _secret("KAGGLE_API_TOKEN")
username, key = _secret("KAGGLE_USERNAME"), _secret("KAGGLE_KEY")
drive_token = Path("/content/drive/MyDrive/kaggle_access_token.txt")
drive_json = Path("/content/drive/MyDrive/kaggle.json")

if api_token:
    os.environ["KAGGLE_API_TOKEN"] = api_token
    print("Kaggle credentials: KAGGLE_API_TOKEN Colab Secret")
elif username and key:
    os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = username, key
    print("Kaggle credentials: KAGGLE_USERNAME/KAGGLE_KEY Colab Secrets (legacy)")
elif drive_token.exists():
    shutil.copy(drive_token, kaggle_dir / "access_token")
    (kaggle_dir / "access_token").chmod(0o600)
    print("Kaggle credentials: MyDrive/kaggle_access_token.txt")
elif drive_json.exists():
    shutil.copy(drive_json, kaggle_dir / "kaggle.json")
    (kaggle_dir / "kaggle.json").chmod(0o600)
    print("Kaggle credentials: MyDrive/kaggle.json (legacy)")
else:
    raise RuntimeError(
        "No Kaggle credentials found. Add a KAGGLE_API_TOKEN Colab Secret (recommended), "
        "or KAGGLE_USERNAME + KAGGLE_KEY Colab Secrets, or upload kaggle_access_token.txt "
        "/ kaggle.json to MyDrive root."
    )

In [ ]:
# ---- clone repo + install deps ----
!rm -rf /content/repo && git clone {REPO_URL} /content/repo
%pip install -q "ultralytics>=8.4.0" "kaggle>=2.2"

In [ ]:
# ---- download competition data (not redistributable -> always pulled from Kaggle) ----
!kaggle competitions download -c severstal-steel-defect-detection -p /content/severstal/raw
!unzip -q -o /content/severstal/raw/severstal-steel-defect-detection.zip -d /content/severstal/raw
!ls /content/severstal/raw

In [ ]:
# ---- convert RLE -> YOLO-seg with the exact same script used locally ----
!python /content/repo/scripts/convert_severstal_to_yolo.py \
    --raw-dir /content/severstal/raw \
    --out-dir /content/severstal/yolo \
    --reports-dir /content/reports \
    --seed {SEED}

In [ ]:
# ---- train (auto-resumes from Drive checkpoint if the session died mid-run) ----
from pathlib import Path
from ultralytics import YOLO

run_dir = Path(DRIVE_ROOT) / "runs" / RUN_NAME
last_ckpt = run_dir / "weights" / "last.pt"

if last_ckpt.exists():
    print(f"Resuming from {last_ckpt}")
    model = YOLO(str(last_ckpt))
    results = model.train(resume=True)
else:
    model = YOLO("yolo26s-seg.pt")
    results = model.train(
        data="/content/severstal/yolo/data.yaml",
        imgsz=IMGSZ,
        epochs=EPOCHS,
        patience=20,
        batch=32,          # tuned for A100 (40GB); auto (-1) under-used A100 memory. Lower this if training on a smaller GPU (e.g. T4/L4).
        seed=SEED,
        project=str(run_dir.parent),
        name=RUN_NAME,
        exist_ok=True,
        # steel-strip-specific augmentation
        degrees=0.0,       # orientation is fixed on the production line
        fliplr=0.5,
        flipud=0.5,        # vertical flip is physically plausible for strips
        copy_paste=0.3,    # seg-only aug, helps rare classes
        hsv_h=0.0,         # images are effectively grayscale
        hsv_s=0.0,
        hsv_v=0.4,
    )

In [ ]:
# ---- final validation with best.pt ----
from ultralytics import YOLO

best_ckpt = run_dir / "weights" / "best.pt"
metrics = YOLO(str(best_ckpt)).val(data="/content/severstal/yolo/data.yaml", imgsz=IMGSZ)
print(f"mask mAP50    : {metrics.seg.map50:.4f}")
print(f"mask mAP50-95 : {metrics.seg.map:.4f}")

In [ ]:
# ---- copy final artifacts to a stable Drive location ----
import shutil
from pathlib import Path

final_dir = Path(DRIVE_ROOT) / "final"
final_dir.mkdir(parents=True, exist_ok=True)
for artifact in ("weights/best.pt", "results.csv", "args.yaml"):
    src = run_dir / artifact
    if src.exists():
        shutil.copy2(src, final_dir / src.name)
        print(f"copied {src} -> {final_dir / src.name}")
print("Done. Download best.pt into the repo's weights/ folder for Phase 2.")